Finding best hyperparameters

In [3]:
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
import pandas as pd
from sklearn.preprocessing import LabelEncoder,label_binarize
from sklearn.metrics import matthews_corrcoef,f1_score,roc_auc_score,make_scorer
import numpy as np

data=pd.read_csv('yeast.csv')
X=data.drop('name',axis=1).to_numpy()
y=data['name'].to_numpy()
le=LabelEncoder()
y=le.fit_transform(y)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42,stratify=y)


In [8]:
# Scorer for multiclass ROC AUC (one-vs-rest)
roc_scorer = make_scorer(roc_auc_score, needs_proba=True, multi_class='ovr', average='macro')

# Pipeline (DecisionTree doesn't need scaling but pipeline keeps preprocessing inside CV)
pipe = Pipeline([
    ('clf', DecisionTreeClassifier(random_state=42))
])

# Parameter grid for DecisionTree
param_grid = {
    'clf__criterion': [ 'entropy'],
    'clf__max_depth': [1,2,3,4,5,6],
    'clf__min_samples_split': [2,3,4,5,6,7],
    'clf__min_samples_leaf': [2,3,4,5,6,7],
    'clf__class_weight': ['balanced',None]
}

cv = StratifiedKFold(n_splits=4, shuffle=True, random_state=42)

grid = GridSearchCV(pipe, param_grid, scoring='roc_auc_ovr', cv=cv, n_jobs=-1, verbose=1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV ROC AUC (macro):", grid.best_score_)

# Evaluate on holdout test set
best = grid.best_estimator_
y_score = best.predict_proba(X_test)
y_test_bin = label_binarize(y_test, classes=range(len(np.unique(y))))
roc_auc_test = roc_auc_score(y_test_bin, y_score, average='macro', multi_class='ovr')
print("Test ROC AUC (macro):", roc_auc_test)



Fitting 4 folds for each of 432 candidates, totalling 1728 fits
Best params: {'clf__class_weight': 'balanced', 'clf__criterion': 'entropy', 'clf__max_depth': 5, 'clf__min_samples_leaf': 4, 'clf__min_samples_split': 2}
Best CV ROC AUC (macro): 0.8140582047641411
Test ROC AUC (macro): 0.747092672887695


In [9]:
clf=DecisionTreeClassifier(random_state=42,criterion= 'entropy', max_depth=5, min_samples_leaf= 4, min_samples_split=2,class_weight='balanced')
y_pred = clf.fit(X_train, y_train).predict(X_test)
# Predictions (probabilities)
y_score = clf.predict_proba(X_test)
# Binarize test labels
y_test_bin = label_binarize(y_test, classes=range(len(le.classes_)))

#Matthews correlation coeffficient
mcc = matthews_corrcoef(y_test, y_pred)
print("MCC:",mcc)
#f1 score
f1 = f1_score(y_test, y_pred,average='weighted')
print("F1 Score:", f1)
#ROC_AUC
roc_auc = roc_auc_score(y_test_bin, y_score, average='macro', multi_class='ovr')
print("ROC AUC Score:", roc_auc)

MCC: 0.39666489345168576
F1 Score: 0.517663482459318
ROC AUC Score: 0.747092672887695
